In [7]:
import pandas as pd
import snowflake.connector
from dotenv import load_dotenv
import os

load_dotenv()

con = snowflake.connector.connect(
    account=os.getenv("SNOWFLAKE_ACCOUNT"),
    user=os.getenv("SNOWFLAKE_USER"),
    password=os.getenv("SNOWFLAKE_PASSWORD"),
    warehouse="COMPUTE_WH",
    database="RIDESHARE",
    schema="MARTS"
)
print("Connected to Snowflake")

Connected to Snowflake


In [8]:
def query(sql):
    cursor = con.cursor()
    cursor.execute(sql)
    cols = [col[0] for col in cursor.description]
    rows = cursor.fetchall()
    return pd.DataFrame(rows, columns=cols)

In [9]:
import plotly.express as px

## Uber vs Lyft - Volume and Revenue Comparison

Which company dominates NYC rideshare, and by how much?

In [15]:
df = query("""
    select
        company,
        count(*) as total_trips,
        round(sum(total_fare), 2) as gross_revenue,
        round(sum(total_fare - driver_pay), 2) as platform_revenue,
        round(avg(total_fare), 2) as avg_fare_per_trip
    from fct_trips
    group by company
""")

df['GROSS_REVENUE'] = df['GROSS_REVENUE'].map('${:,.2f}'.format)
df['PLATFORM_REVENUE'] = df['PLATFORM_REVENUE'].map('${:,.2f}'.format)
df

,COMPANY,TOTAL_TRIPS,GROSS_REVENUE,PLATFORM_REVENUE,AVG_FARE_PER_TRIP
0,Lyft,4898054,"$118,548,815.74","$45,348,420.75",24.20
1,Uber,13564036,"$356,808,525.53","$120,105,305.74",26.31


Analysis shows Uber completely dominating NYC rideshare, accounting for 73% of all rides with riders paying an average of $26.31 per Uber trip compared to $24.20 on Lyft. Although, when it comes to revenue percentage, Lyft keeps a 38% margin while Uber sits at 33.7%. Regardless, Uber is certainly controlling the majority of rides in New York. Brand recognition certainly plays a factor in that success. Uber launched two years before Lyft, expanded internationally far earlier, and evolved beyond rides entirely with Uber Eats — meaning customers don't need to request a ride to stay engaged with the app.